# Equatorial particle estimation

This notebook will be used to estimate a particle's Euler angles in the equatorial plane. That is:
1. We don't know the particle's rotation in the membrane, so we don't know phi
2. Since all particles we're picking are in the equatorial plane, we can assume theta = pi/2
3. We can estimate psi by looking at the particle's location relative to the vesicle's centroid

We want to then convert these Euler angles to the axis-angle representation, consistent with the cryoSPARC file format.

## Procedure
1. Load the particles, the micrograph, and the vesicle masks
2. For each particle, find the vesicle it is in--we do this by finding the vesicle with the highest percentage of the particle's area
    > We will add a tolerance of 30 pixels to a vesicle's centroid, if outside this tolerance, we discard the particle.
    
    > Additionally, we will also discard particles that are not in the equatorial plane, so particles that are inside the binary mask of the vesicle
3. For each particle, we will then estimate the alpha angle, according to the methodology proposed by Sigworth et al., 2014
4. We will then convert the Euler angles to the axis-angle representation, and save the results to a new .cs file

### Step 0: Import libraries

In [1]:
import os
import matplotlib.pyplot as plt
import numpy as np
import random
import mrcfile
import pickle

from cryosparc.dataset import Dataset
import numpy as np
import os
import matplotlib.pyplot as plt
from scipy import ndimage
from scipy.spatial.transform import Rotation as R

import math
import numpy as np
import torch
import torch.nn.functional as F
from collections import defaultdict

### Step 1: Load the particles in

In [2]:
cs_file = '/Users/gerdbizi/Desktop/Rubinstein Lab/ryan_images/J3365/extracted_particles.cs'
binarized_micrograph_path = '/Users/gerdbizi/Desktop/Rubinstein Lab/filtered_vesicles'
cs = Dataset.load(cs_file)
scaling_factor = 1024/4096

print(cs) # Print the fields of the dataset

Dataset([  # 198766 items, 32 fields
    ('uid', [ 3594195723540814930 7824494210550147048 14101447717293291797 ... 17039687655059442827 2065012062762269259 1275955471049284477]),
    ('blob/path', ['J3365/extract/FoilHole_10169637_Data_10173120_10173122_20231010_104011_EER_patch_aligned_doseweighted_particles.mrc' 'J3365/extract/FoilHole_10169637_Data_10173120_10173122_20231010_104011_EER_patch_aligned_doseweighted_particles.mrc' 'J3365/extract/FoilHole_10169637_Data_10173120_10173122_20231010_104011_EER_patch_aligned_doseweighted_particles.mrc' ... 'J3365/extract/FoilHole_12287403_Data_10173110_10173112_20231012_124321_EER_patch_aligned_doseweighted_particles.mrc' 'J3365/extract/FoilHole_12287403_Data_10173110_10173112_20231012_124321_EER_patch_aligned_doseweighted_particles.mrc' 'J3365/extract/FoilHole_12287403_Data_10173110_10173112_20231012_124321_EER_patch_aligned_doseweighted_particles.mrc']),
    ('blob/idx', [0 1 2 ... 5 6 7]),
    ('blob/shape', [[500 500] [500 500] [500 500]

### Step 2: Figure out which vesicle a particle is in

To figure out which vesicle a particle is in, we need to find the vesicle with the highest percentage of the particle's area.

So, we need to define a ```percentage_of_particle_area``` function.

In [3]:
def percent_particle_in_vesicle(particle_box, vesicle_box):
    """
    Determine the percentage of the particle that is in the vesicle
    :param particle_box: The bounding box of the particle
    :param vesicle_box: The bounding box of the vesicle
    :return: The percentage of the particle that is in the vesicle
    """
    # print(particle_box, vesicle_box)
    particle_top_x, particle_top_y, particle_bottom_x, particle_bottom_y = particle_box
    vesicle_top_x, vesicle_top_y, vesicle_bottom_x, vesicle_bottom_y = vesicle_box

    particle_area = (particle_bottom_x - particle_top_x) * (particle_bottom_y - particle_top_y)

    # The area of the intersection of the particle and the vesicle
    intersection_area = max(0, min(particle_bottom_x, vesicle_bottom_x) - max(particle_top_x, vesicle_top_x)) * \
                        max(0, min(particle_bottom_y, vesicle_bottom_y) - max(particle_top_y, vesicle_top_y))

    return intersection_area / particle_area

To also account for our tolerance, we need to define a function that returns the radius of a vesicle and a function that returns the centroid of a vesicle.

In [4]:
def vesicle_radius(vesicle):
    """
    Return the radius of a vesicle
    :param vesicle: The vesicle
    :return: The radius of the vesicle
    """
    vesicle_mask = vesicle['segmentation'] > 0
    area = ndimage.sum(vesicle_mask)
    return np.sqrt(area / np.pi)

def get_centroid(vesicle):
    """
    Get the centroid of the vesicle via the centre of mass
    :param vesicle: The vesicle
    :return: The centroid of the vesicle in (x, y)
    """
    y, x = ndimage.measurements.center_of_mass(vesicle['segmentation'])
    return x, y

def in_vesicle(vesicle, x, y):
    """
    Check if a particle is in a vesicle
    :param particle: The particle
    :param vesicle: The vesicle
    :return: True if the particle is in the vesicle, False otherwise
    """
    return vesicle['segmentation'][x, y] != 0

Then, we also need to define a ```which_vesicle``` function.

In [5]:
def which_vesicle(x, y, shape_x, shape_y, micrograph, radius_tolerance=10):
    """
    Determine which vesicle the particle is in.
    :param x: The x coordinate of the particle
    :param y: The y coordinate of the particle
    :param micrograph: The micrograph
    :param radius_tolerance: The tolerance in pixels for the vesicle's centroid
    :return: The vesicle ID
    """
    vesicle = None
    max_percent_particle_in_vesicle = 0

    particle_top_x = int(x - shape_x//2)
    particle_top_y = int(y - shape_y//2)
    particle_bottom_x = int(x + shape_x//2)
    particle_bottom_y = int(y + shape_y//2)
    particle_box = [particle_top_x, particle_top_y, particle_bottom_x, particle_bottom_y]

    vesicle_top_x = None
    vesicle_top_y = None
    vesicle_bottom_x = None
    vesicle_bottom_y = None

    for entry in micrograph['masks']:
        vesicle_top_x = int(entry['bbox'][0])
        vesicle_top_y = int(entry['bbox'][1])
        vesicle_bottom_x = int(entry['bbox'][2]) + vesicle_top_x
        vesicle_bottom_y = int(entry['bbox'][3]) + vesicle_top_y
        vesicle_box = [vesicle_top_x, vesicle_top_y, vesicle_bottom_x, vesicle_bottom_y]

        percent = percent_particle_in_vesicle(particle_box, vesicle_box)
        if percent > max_percent_particle_in_vesicle:
            max_percent_particle_in_vesicle = percent
            vesicle = entry

    if not vesicle:
        return None
    
    prime_key_uint64 = np.uint64(vesicle['prime_key'])
    segmentation_mask = (micrograph['composite_mask'] % prime_key_uint64) == 0
    del micrograph
    vesicle['segmentation'] = segmentation_mask.astype(np.uint8)
    
    # Check if the particle is in the vesicle, if it is, the particle is not in the equatorial plane
    if in_vesicle(vesicle, x, y):
        return None

    # Check if the particle is within the vesicle's radius
    radius = vesicle_radius(vesicle)
    if radius < 0:
        return None
    
    particle_centroid = (x, y)
    vesicle_centroid = get_centroid(vesicle)
    distance = np.linalg.norm(np.array(particle_centroid) - np.array(vesicle_centroid))
    
    if distance > radius:
        return None

    return vesicle

### Step 3: Estimate Theta and Psi

Since we know that all particles are in the equatorial plane, we can set theta = pi/2.

We can then estimate psi by looking at the particle's location relative to the vesicle's centroid.

Sigworth et al. (2014) noted the following conversions:
phi = gamma - pi/2
theta = beta
psi = alpha + pi/2

So, we can only estimate alpha and beta based on the particle's location relative to the vesicle's centroid.

In [6]:
def get_alpha(reference_point, x, y):
    """
    Calculate angle from positive y-axis (clockwise positive) for a vector in image coordinates.
    Specifically designed for particle pose estimation in cryo-EM.
    
    Args:
        reference_point (tuple): (x,y) coordinates of particle centroid
        x (float): x-coordinate of target point (e.g., particle peak or feature)
        y (float): y-coordinate of target point
    
    Returns:
        float: Angle in radians, range [-π, π], clockwise positive from y-axis.
             Compatible with cryoSPARC pose conventions.
    """
    # Convert to vector from centroid
    dx = x - reference_point[0]
    # Invert dy for image coordinates (y increases downward)
    dy = -(y - reference_point[1])
    
    # Use atan2 to get angle from x-axis (standard mathematical angle)
    standard_angle = math.atan2(dy, dx)
    
    # Convert to our convention:
    # 1. Subtract π/2 to make 0° at positive y-axis
    alpha = standard_angle - math.pi/2
    
    # 2. Negate for clockwise positive convention
    alpha = -alpha
    
    # Normalize to [-π, π] range
    if alpha > math.pi:
        alpha -= 2*math.pi
    elif alpha < -math.pi:
        alpha += 2*math.pi
        
    return alpha

def sigworth_to_euler(sigworth_angles):
    gamma, beta, alpha = sigworth_angles
    phi = gamma - np.pi/2
    theta = beta
    psi = alpha + np.pi/2

    if psi > np.pi:
        psi = psi - 2*np.pi
    elif psi < -np.pi:
        psi = psi + 2*np.pi

    psi += np.pi

    return phi, theta, psi


Now, we will also include the pertinent function for converting from Euler angles to axis-angle representation. The functions are taken from: https://github.com/asarnow/pyem/blob/master/pyem/geom/convert.py

We need to go from: Euler -> Rotation Matrix -> Quaternion -> Axis-Angle

We also need to define the inverse functions: Axis-Angle -> Quaternion -> Rotation Matrix -> Euler

In [7]:
def euler2rot(alpha, beta, gamma):
    ca = np.cos(alpha)
    cb = np.cos(beta)
    cg = np.cos(gamma)
    sa = np.sin(alpha)
    sb = np.sin(beta)
    sg = np.sin(gamma)
    cc = cb * ca
    cs = cb * sa
    sc = sb * ca
    ss = sb * sa
    r = np.array([[cg * cc - sg * sa, cg * cs + sg * ca, -cg * sb],
                  [-sg * cc - cg * sa, -sg * cs + cg * ca, sg * sb],
                  [sc, ss, cb]])
    return r

def rot2quat(r):
    q = np.zeros(4, dtype=r.dtype)
    tr = np.trace(r)
    if tr > 0:
        q[0] = np.sqrt(tr + 1) / 2
        sinv = 1 / (q[0] * 4)
        q[1] = sinv * (r[1, 2] - r[2, 1])
        q[2] = sinv * (r[2, 0] - r[0, 2])
        q[3] = sinv * (r[0, 1] - r[1, 0])
    else:
        mi = np.argmax(np.diag(r))
        if mi == 0:
            q[1] = np.sqrt(r[0, 0] - r[1, 1] - r[2, 2] + 1) / 2
            sinv = 1 / (q[1] * 4)
            q[0] = sinv * (r[1, 2] - r[2, 1])
            q[2] = sinv * (r[0, 1] + r[1, 0])
            q[3] = sinv * (r[0, 2] + r[2, 0])
        elif mi == 1:
            q[2] = np.sqrt(r[1, 1] - r[2, 2] - r[0, 0] + 1) / 2
            sinv = 1 / (q[2] * 4)
            q[0] = sinv * (r[2, 0] - r[0, 2])
            q[1] = sinv * (r[0, 1] + r[1, 0])
            q[3] = sinv * (r[1, 2] + r[2, 1])
        else:
            q[3] = np.sqrt(r[2, 2] - r[0, 0] - r[1, 1] + 1) / 2
            sinv = 1 / (q[3] * 4)
            q[0] = sinv * (r[0, 1] - r[1, 0])
            q[1] = sinv * (r[0, 2] + r[2, 0])
            q[2] = sinv * (r[1, 2] + r[2, 1])
    return q

def quat2aa(q):
    n = np.linalg.norm(q[1:])
    ax = q[1:] / n if n > 0 else np.zeros(3, dtype=q.dtype)
    theta = 2 * np.arctan2(n, q[0])  # Or 2 * np.arccos(q[0])
    return theta * ax

def aa2quat(ax, theta=None):
    if theta is None:
        theta = np.linalg.norm(ax)
        if theta != 0:
            ax = ax / theta
    q = np.zeros(4, dtype=ax.dtype)
    q[0] = np.cos(theta / 2)
    q[1:] = ax * np.sin(theta / 2)
    return q

def quat2rot(q):
    n = np.sum(q**2)
    s = 0 if n == 0 else 2 / n
    wx = s * q[0] * q[1]
    wy = s * q[0] * q[2]
    wz = s * q[0] * q[3]
    xx = s * q[1] * q[1]
    xy = s * q[1] * q[2]
    xz = s * q[1] * q[3]
    yy = s * q[2] * q[2]
    yz = s * q[2] * q[3]
    zz = s * q[3] * q[3]
    r = np.array([[1 - (yy + zz), xy + wz,       xz - wy],
                  [xy - wz,       1 - (xx + zz), yz + wx],
                  [xz + wy,       yz - wx,       1 - (xx + yy)]], dtype=q.dtype)
    return r

def rot2euler(r):
    """Decompose rotation matrix into Euler angles"""
    # assert(isrotation(r))
    # Shoemake rotation matrix decomposition algorithm with same conventions as Relion.
    epsilon = np.finfo(np.double).eps
    abs_sb = np.sqrt(r[0, 2] ** 2 + r[1, 2] ** 2)
    if abs_sb > 16 * epsilon:
        gamma = np.arctan2(r[1, 2], -r[0, 2])
        alpha = np.arctan2(r[2, 1], r[2, 0])
        if np.abs(np.sin(gamma)) < epsilon:
            sign_sb = np.sign(-r[0, 2]) / np.cos(gamma)
        else:
            sign_sb = np.sign(r[1, 2]) if np.sin(gamma) > 0 else -np.sign(r[1, 2])
        beta = np.arctan2(sign_sb * abs_sb, r[2, 2])
    else:
        if np.sign(r[2, 2]) > 0:
            alpha = 0
            beta = 0
            gamma = np.arctan2(-r[1, 0], r[0, 0])
        else:
            alpha = 0
            beta = np.pi
            gamma = np.arctan2(r[1, 0], -r[0, 0])
    return alpha, beta, gamma


Now, let's proceed with filtering for equatorial particles.

In [8]:
equatorial_particle_data = []  # Store minimal necessary data

for i, particle in enumerate(cs.rows()):
    # Extract location data
    try:
        shape_x, shape_y = particle['blob/shape']
        shape_x, shape_y = int(shape_x * scaling_factor), int(shape_y * scaling_factor)
        micrograph_uid = particle['location/micrograph_uid']
        mic_shape_x, mic_shape_y = particle['location/micrograph_shape']
        mic_shape_x, mic_shape_y = int(mic_shape_x * scaling_factor), int(mic_shape_y * scaling_factor)
        centre_x_frac = particle['location/center_x_frac']
        centre_y_frac = particle['location/center_y_frac']
        # shift_x, shift_y = particle['alignments3D/shift']
    
        # Calculate new location
        # new_loc_x = int((centre_x_frac * mic_shape_x - shift_x))
        # new_loc_y = int((centre_y_frac * mic_shape_y - shift_y))
        new_loc_x = int(centre_x_frac * mic_shape_x)
        new_loc_y = int(centre_y_frac * mic_shape_y)
        
        # Load and check micrograph
        micrograph_id = f'/Users/gerdbizi/Desktop/Rubinstein Lab/filtered_vesicles/{micrograph_uid}_vesicles_filtered.pkl'
        get_micrograph = lambda micrograph_id: pickle.load(open(micrograph_id, 'rb'))
        micrograph = get_micrograph(micrograph_id)

        if micrograph is None:
            continue
        
        # Process vesicle
        vesicle = which_vesicle(new_loc_x, new_loc_y, shape_x, shape_y, micrograph)
        if vesicle is None:
            del micrograph
            continue
   
        centroid = get_centroid(vesicle)
        equatorial_particle_data.append({
            'particle': particle,
            'new_loc': (new_loc_x, new_loc_y),
            'centroid': centroid
        })
        
        if i % 100 == 0:
            print(f"Processed {i} particles, found {len(equatorial_particle_data)} equatorial particles")
            
    except Exception as e:
        print(f"Error processing particle {i}: {str(e)}")
        continue

print(f"Total equatorial particles found: {len(equatorial_particle_data)} from {len(cs)} particles")

/var/folders/hy/7f2mb4w14q56bjcr44s2483m0000gn/T/ipykernel_54404/327977166.py:17: DeprecationWarning: Please import `center_of_mass` from the `scipy.ndimage` namespace; the `scipy.ndimage.measurements` namespace is deprecated and will be removed in SciPy 2.0.0.
  y, x = ndimage.measurements.center_of_mass(vesicle['segmentation'])


Error processing particle 210: [Errno 2] No such file or directory: '/Users/gerdbizi/Desktop/Rubinstein Lab/filtered_vesicles/7942296721786380808_vesicles_filtered.pkl'
Error processing particle 211: [Errno 2] No such file or directory: '/Users/gerdbizi/Desktop/Rubinstein Lab/filtered_vesicles/7942296721786380808_vesicles_filtered.pkl'
Error processing particle 212: [Errno 2] No such file or directory: '/Users/gerdbizi/Desktop/Rubinstein Lab/filtered_vesicles/7942296721786380808_vesicles_filtered.pkl'
Error processing particle 213: [Errno 2] No such file or directory: '/Users/gerdbizi/Desktop/Rubinstein Lab/filtered_vesicles/7942296721786380808_vesicles_filtered.pkl'
Error processing particle 214: [Errno 2] No such file or directory: '/Users/gerdbizi/Desktop/Rubinstein Lab/filtered_vesicles/7942296721786380808_vesicles_filtered.pkl'
Error processing particle 215: [Errno 2] No such file or directory: '/Users/gerdbizi/Desktop/Rubinstein Lab/filtered_vesicles/7942296721786380808_vesicles_

Now, let's estimate the alpha angle for each particle.


In [10]:
# # First create a new dictionary with all the same fields as original particles
# new_angle_data = defaultdict(list)
# estimated_angles = []

# # Process particles one at a time
# for data in equatorial_particle_data:
#     particle = data['particle']
#     new_loc_x, new_loc_y = data['new_loc']
#     centroid = data['centroid']
    
#     # Calculate new pose
#     alpha_sigworth = get_alpha(centroid, new_loc_x, new_loc_y)
#     sigworth_angles = [0, np.pi/2, alpha_sigworth]
#     predicted_euler_angles = sigworth_to_euler(sigworth_angles)
#     print(predicted_euler_angles)

#     # Convert the native axis-angle in cryoSPARC to Euler angles
#     cs_quat = aa2quat(particle['alignments3D/pose'])
#     cs_rot = quat2rot(cs_quat)
#     cs_euler = rot2euler(cs_rot)
#     print(cs_euler)

#     cs_euler = [a for a in cs_euler]

#     cs_euler[1] = predicted_euler_angles[1]
#     cs_euler[2] = predicted_euler_angles[2]

#     estimated_angles.append(cs_euler)

#     # Convert the Euler angles back to axis-angle representation
#     cs_rot = euler2rot(*cs_euler)
#     cs_quat = rot2quat(cs_rot)
#     cs_aa = quat2aa(cs_quat)

#     # Convert axis angle values from double to float
#     axis_angle = cs_aa.astype(np.float32)
    
#     # Add all fields from original particle to our new dictionary
#     for field, value in particle.items():
#         if field == 'alignments3D/pose':
#             new_angle_data[field].append(axis_angle)  # Use our new calculated pose
#         else:
#             new_angle_data[field].append(value)  # Keep original value for all other fields
            
#     if len(new_angle_data['uid']) % 100 == 0:
#         print(f"Processed {len(new_angle_data['uid'])} particles")

# # Convert to regular dictionary and create Dataset
# final_dataset = dict(new_angle_data)
# dataset_with_angle = Dataset(final_dataset)

# # Save the new dataset in a new cs file
# dataset_with_angle.save('/Users/gerdbizi/Desktop/Rubinstein Lab/rotation_debugging_495/angle_estimated_equatorial_particles.cs')
# print("Saved dataset with new angles")


Now an attempt using rotation matrices instead of Euler angles.

In [9]:
import pytorch3d.transforms as transforms

# First create a new dictionary with all the same fields as original particles
new_angle_data = defaultdict(list)
estimated_angles = []

# Process particles one at a time
for data in equatorial_particle_data:
    particle = data['particle']
    new_loc_x, new_loc_y = data['new_loc']
    centroid = data['centroid']
    
    # Calculate new pose
    alpha_sigworth = get_alpha(centroid, new_loc_x, new_loc_y)
    
    # Get random value between 0 and 2*pi
    random_value = np.random.uniform(0, 2*np.pi)
    euler_angles = [random_value, np.pi/2, -(alpha_sigworth + np.pi/2)]

    # Create tensor of angles
    euler_angles = torch.tensor(euler_angles)

    # Create rotation matrix for desired orientation
    desired_rotation = transforms.euler_angles_to_matrix(euler_angles, 'ZYZ')
    
    aa = transforms.matrix_to_axis_angle(desired_rotation)
    
    # Add all fields from original particle to our new dictionary
    for field, value in particle.items():
        if field == 'alignments3D/pose':
            new_angle_data[field].append(aa)
        else:
            new_angle_data[field].append(value)
            
    if len(new_angle_data['uid']) % 100 == 0:
        print(f"Processed {len(new_angle_data['uid'])} particles")

# Convert to regular dictionary and create Dataset
final_dataset = dict(new_angle_data)
dataset_with_angle = Dataset(final_dataset)

# Save the new dataset in a new cs file
dataset_with_angle.save('/Users/gerdbizi/Desktop/Rubinstein Lab/rotation_debugging_495/angle_estimated_equatorial_particles_rotation_matrix_big.cs')
print("Saved dataset with new angles of length")


Processed 100 particles
Processed 200 particles
Processed 300 particles
Processed 400 particles
Processed 500 particles
Processed 600 particles
Processed 700 particles
Processed 800 particles
Processed 900 particles
Processed 1000 particles
Processed 1100 particles
Processed 1200 particles
Processed 1300 particles
Processed 1400 particles
Processed 1500 particles
Processed 1600 particles
Processed 1700 particles
Processed 1800 particles
Processed 1900 particles
Processed 2000 particles
Processed 2100 particles
Processed 2200 particles
Processed 2300 particles
Processed 2400 particles
Processed 2500 particles
Processed 2600 particles
Processed 2700 particles
Processed 2800 particles
Processed 2900 particles
Processed 3000 particles
Processed 3100 particles
Processed 3200 particles
Processed 3300 particles
Processed 3400 particles
Processed 3500 particles
Processed 3600 particles
Processed 3700 particles
Processed 3800 particles
Processed 3900 particles
Processed 4000 particles
Processed